* Date : 17-Nov-2025
* SCD-2 Implementation Using Delta Approach
* Using MySQL as DB , we can use Snowflake as well in a similar fashion 
#### Notes : 
* Issues : Mostly the version mis-match between Spark Version Vs Java Vs mysql-connector 


In [11]:
"""

💾  ⚡   🎉   📥   📤   🔧   ❌   ✅   ⭐   🔥

"""

print("Saved symbols ⚡ 🎉 📥 📤 🔧 ❌ ✅ ⭐ 🔥")

Saved symbols ⚡ 🎉 📥 📤 🔧 ❌ ✅ ⭐ 🔥


In [1]:
#############################################
# 0. Environment Setup
#############################################

import os
import glob

# Ensure Spark uses its own internal installation
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]

# Ensure Java 17 is available
os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.16/libexec/openjdk.jdk/Contents/Home"

In [2]:


#############################################
# 1. Locate MySQL JDBC JAR
#############################################

cwd = os.getcwd()

search_roots = [
    cwd,
    os.path.join(cwd, "jars"),
    os.path.abspath(os.path.join(cwd, "..")),
    os.path.abspath(os.path.join(cwd, "..", "jars")),
    os.path.abspath(os.path.join(cwd, "..", "..")),
    os.path.abspath(os.path.join(cwd, "..", "..", "jars")),
    os.path.expanduser("~/Desktop/TEST_AGAIN/jars"),
]

patterns = ["*mysql*connector*.jar", "mysql-connector*.jar", "*mysql*.jar"]

matches = []
for r in search_roots:
    for p in patterns:
        matches.extend(glob.glob(os.path.join(r, "**", p), recursive=True))

matches = sorted(set(matches))
mysql_jar = matches[0] if matches else None

if not mysql_jar:
    raise FileNotFoundError("❌ MySQL JDBC connector not found. Place jar under ./jars/")

print("✅ Found MySQL JDBC:", mysql_jar)

✅ Found MySQL JDBC: /Users/mukesh/Desktop/TEST_AGAIN/jars/mysql-connector-j-9.4.0.jar


In [3]:


#############################################
# 2. Initialize Spark with Delta Extensions
#############################################

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName("Delta_SCD")
    .config("spark.jars", mysql_jar)
    .config("spark.driver.extraClassPath", mysql_jar)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("🔧 Spark Version:", spark.version)
print("🔧 Delta Version Loaded")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/17 18:03:07 WARN Utils: Your hostname, mukeshs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.221 instead (on interface en0)
25/11/17 18:03:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mukesh/Desktop/TEST_AGAIN/spark_sf_env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mukesh/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mukesh/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7eea5ccf-e86b-499d-b314-61c58004dfde;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 95ms :: 

🔧 Spark Version: 4.0.1
🔧 Delta Version Loaded


In [4]:


#############################################
# 3. MySQL Connection Helper
#############################################

from mysql_spark import ConnectDB

HOST = "127.0.0.1"
USER = "root"
PASSWORD = "Paridhi@2019#"
DATABASE = "dw_poc"

db = ConnectDB(host=HOST, user=USER, password=PASSWORD, database=DATABASE)

Initialising the database configuration...


In [5]:

#############################################
# 4. Load Source and Target Tables from MySQL
#############################################

print("\n📥 Loading Source Table (employee_src)")
df_src = db.read_mysql_table_to_spark_df(spark, "employee_src")
df_src.show()

print("\n📥 Loading Target Table (employee_trg)")
df_trg = db.read_mysql_table_to_spark_df(spark, "employee_trg")
df_trg.show()


📥 Loading Source Table (employee_src)

--- Spark Read: Reading Entire Table 'employee_src' ---
Successfully read MySQL data into Spark DataFrame. Schema:
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)

+------+-------+----------+------+--------------+
|emp_id|   name|department|salary|effective_date|
+------+-------+----------+------+--------------+
|     1| MUKESH|        HR|  5000|    2025-10-30|
|     2|   YASH|        IT|  6000|    2025-10-30|
|     3|CHARLIE|   FINANCE|  8000|    2025-10-30|
+------+-------+----------+------+--------------+


📥 Loading Target Table (employee_trg)

--- Spark Read: Reading Entire Table 'employee_trg' ---
Successfully read MySQL data into Spark DataFrame. Schema:
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary

In [6]:

#############################################
# 5. Write Target Table into Delta Format
#############################################

delta_path = "/tmp/employee_trg_delta"

print("\n💾 Writing MySQL target table into Delta:", delta_path)
df_trg.write.format("delta").mode("overwrite").save(delta_path)


💾 Writing MySQL target table into Delta: /tmp/employee_trg_delta


25/11/17 18:03:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:


#############################################
# 6. Perform Delta Merge (SCD Type-2)
#############################################

from delta.tables import DeltaTable

delta_trg = DeltaTable.forPath(spark, delta_path)

print("\n⚡ Running Delta MERGE (SCD Type-2)")

delta_trg.alias("t").merge(
    df_src.alias("s"),
    "t.emp_id = s.emp_id AND t.flag = true"
).whenMatchedUpdate(
    condition="t.name != s.name OR t.department != s.department OR t.salary != s.salary",
    set={
        "effective_end_date": current_date(),
        "flag": lit(False)
    }
).whenNotMatchedInsert(
    values={
        "emp_id": "s.emp_id",
        "name": "s.name",
        "department": "s.department",
        "salary": "s.salary",
        "effective_start_date": current_date(),
        "effective_end_date": lit("9999-12-31").cast(DateType()),
        "flag": lit(True)
    }
).execute()

print("✅ MERGE Completed")


⚡ Running Delta MERGE (SCD Type-2)
✅ MERGE Completed


25/11/17 18:04:34 WARN MapPartitionsRDD: RDD 43 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [8]:

#############################################
# 7. Load Merged Delta Table & Write Back to MySQL
#############################################

df_final = spark.read.format("delta").load(delta_path)

print("\n📤 Writing Final Merged Data Back to MySQL: employee_trg_final")

db.write_spark_data_to_mysql(df_final, "employee_trg_final")

print("🎉 Pipeline Completed Successfully!")



📤 Writing Final Merged Data Back to MySQL: employee_trg_final

--- Spark Write: Writing to 'employee_trg_final' in 'append' mode ---
Successfully wrote Spark DataFrame to MySQL table 'employee_trg_final'.
🎉 Pipeline Completed Successfully!


In [13]:
import sys
print(sys.executable)

/Users/mukesh/Desktop/TEST_AGAIN/spark_sf_env/bin/python
